# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [50]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
import re
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemma3:270m'
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [90]:
links = fetch_website_links("https://onealbum.app")
links

['#main-content',
 '#top',
 '#how-it-works',
 '#gallery',
 'https://play.google.com/store/apps/details?id=com.joinonealbum.app',
 'https://play.google.com/store/apps/details?id=com.joinonealbum.app',
 '#how-it-works',
 'https://play.google.com/store/apps/details?id=com.joinonealbum.app',
 '#how-it-works',
 '#top',
 'https://play.google.com/store/apps/details?id=com.joinonealbum.app',
 '#how-it-works',
 '#faq',
 '#download',
 '/privacy',
 '/terms',
 'https://instagram.com/joinonealbum',
 'https://x.com/joinonealbum',
 'https://linkedin.com/company/joinonealbum']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [71]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [72]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [91]:
print(get_links_user_prompt("https://onealbum.app"))


Here is the list of links on the website https://onealbum.app -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main-content
#top
#how-it-works
#gallery
https://play.google.com/store/apps/details?id=com.joinonealbum.app
https://play.google.com/store/apps/details?id=com.joinonealbum.app
#how-it-works
https://play.google.com/store/apps/details?id=com.joinonealbum.app
#how-it-works
#top
https://play.google.com/store/apps/details?id=com.joinonealbum.app
#how-it-works
#faq
#download
/privacy
/terms
https://instagram.com/joinonealbum
https://x.com/joinonealbum
https://linkedin.com/company/joinonealbum


In [74]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [92]:
select_relevant_links("https://onealbum.app")

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 3 relevant links


{'links': [{'type': 'About page', 'url': 'https://onealbum.app/about'},
  {'type': 'FAQ page', 'url': 'https://onealbum.app/faq'},
  {'type': 'Privacy & Terms page', 'url': 'https://onealbum.app/privacy'}]}

In [76]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [93]:
select_relevant_links("https://onealbum.app")

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 2 relevant links


{'links': ['https://www.onealbum.app/about',
  'https://www.onealbum.app/careers']}

In [94]:
select_relevant_links("https://onealbum.app")

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 1 relevant links


{'links': [{'type': 'store page',
   'url': 'https://play.google.com/store/apps/details?id=com.joinonealbum.app'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [79]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [95]:
print(fetch_page_and_all_relevant_links("https://onealbum.app"))

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 4 relevant links
## Landing Page:

OneAlbum — Shared photo albums for events

Skip to main content
OneAlbum
How it works
Gallery
Menu
0
%
Download OneAlbum
Every photo, in one place
Create a shared album for your event, invite everyone via QR code, and collect all the photos in one place — weddings, trips, parties, and more.
Download OneAlbum
See how it works
Loading...
1
See OneAlbum in action
Scroll down
Every
event
has
a
hundred
cameras.
They're
just
scattered
across
a
hundred
phones.
OneAlbum
brings
them
together
—
one
album,
every
angle,
every
moment.
The
photos
that
would've
stayed
buried
in
someone's
camera
roll,
now
shared
with
everyone
who
was
there.
How it works
Four steps. One album. Every photo from everyone at your event, collected in one place automatically.
0
1
Create in seconds
Start an event album with a name, date, and vibe. Ready to share the moment it's created.
0
2
Invite via QR
Share a QR 

In [96]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [97]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [99]:
get_brochure_user_prompt("OneAlbum", "https://onealbum.app")

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 6 relevant links


"\nYou are looking at a company called: OneAlbum\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nOneAlbum — Shared photo albums for events\n\nSkip to main content\nOneAlbum\nHow it works\nGallery\nMenu\n0\n%\nDownload OneAlbum\nEvery photo, in one place\nCreate a shared album for your event, invite everyone via QR code, and collect all the photos in one place — weddings, trips, parties, and more.\nDownload OneAlbum\nSee how it works\nLoading...\n1\nSee OneAlbum in action\nScroll down\nEvery\nevent\nhas\na\nhundred\ncameras.\nThey're\njust\nscattered\nacross\na\nhundred\nphones.\nOneAlbum\nbrings\nthem\ntogether\n—\none\nalbum,\nevery\nangle,\nevery\nmoment.\nThe\nphotos\nthat\nwould've\nstayed\nburied\nin\nsomeone's\ncamera\nroll,\nnow\nshared\nwith\neveryone\nwho\nwas\nthere.\nHow it works\nFour steps. One album. Every photo from everyone at you

In [100]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [101]:
create_brochure("OneAlbum", "https://onealbum.app")

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 3 relevant links


# OneAlbum: Shared Photo Albums for Events
## Introduction

OneAlbum is a revolutionary app that brings together all the photos from every guest at an event into one shared album. No more scattered across everyone's phones or lost forever.

## How it Works

Our app allows you to create a shared album for your event in seconds, share it with a QR code or join link, and let guests upload their photos and videos straight from their phones. Everyone sees the full album as it fills up, live.

*   Create an album for your event in seconds
*   Share it with a QR code or a simple join link
*   Guests upload their photos and videos straight from their phones — no account hassle
*   Everyone sees the full album as it fills up, live

## Features

### Full Quality Photos

Collect high-quality photos from every guest without blurry, compressed group-chat versions.

### Easy Joining

No more chasing friends for photos after the event! Our app makes it easy to join with a QR code or join link.

### Built for Any Occasion

Whether it's a wedding, birthday, trip, or party, OneAlbum is built for any occasion.

## Benefits

*   **One album, every angle** – collect photos from every guest automatically
*   **No data shared with third parties** – your data is safe and secure.
*   **Easy to use** – no account hassle, just upload and enjoy!

## What Our Customers Say

"OneAlbum is a game-changer for event photographers. It's so easy to use and looks amazing." - Emily R.

"I was skeptical at first, but now I don't know how I ever lived without OneAlbum!" - Mark K.


If you want to start your own shared photo album today, download the app or visit our website to learn more!

You can reach out to us here:
https://www.onalbum.com

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [102]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [103]:
stream_brochure("OneAlbum", "https://onealbum.app")

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 3 relevant links


# OneAlbum Brochure


## Our Mission


At OneAlbum, we believe that every moment is worth capturing and sharing with others. Our mission is to make it easy for people to come together and create a shared album of memories from their events.


## How We Work


We use the power of QR codes and technology to bring all your photos together in one place, so you can share them with your friends and family instantly. Here's how:


*   Create an event album in seconds
*   Share it with a QR code or a simple join link
*   Guests upload their photos and videos straight from their phones — no account hassle
*   Everyone sees the full album as it fills up, live


## Our Focus


Our focus is on making every occasion special by collecting all your photos in one place. Whether it's a wedding, birthday, trip, or party, OneAlbum is here for you.


## What We Love About Our Users


We love seeing how our users showcase their memories with friends and family from around the world.

They use us to:

*   Share milestones
*   Document adventures
*   Celebrate achievements

Every story told in an album showcases what's truly important: making unforgettable memories alongside those who matter most.

Our aim is building strong relationships, fostering trust, promoting connection through simple yet effective ideas.

In [104]:
stream_brochure("OneAlbum", "https://onealbum.app")

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 2 relevant links


# OneAlbum Brochure

## Welcome to OneAlbum

OneAlbum is a revolutionary mobile app designed for capturing every moment of special events. Our mission is to bring everyone at your wedding, birthday, trip, or party together in one shared album, making it easy to relive the memories long after the event ends.

## How It Works

Our app works in just four simple steps:

1. Create an Event Album
   - In seconds, create a new event album with a name, date, and vibe that captures the essence of your occasion.
2. Invite Guests via QR Code or Link
   - Share a unique QR code or invite link for guests to join instantly.

3. Share Memories with Everyone Involved
   - Users simply upload their photos and videos from their camera roll, contributing to one cohesive album that everyone will see in real-time.

4. Relive Together  
   - Keep the full, high-quality album forever, as a cherished keepsake of memories shared across all who participated.

## Why Choose OneAlbum?

• **One Album, Every Perspective**: Capture every moment from every guest, without needing them to join individually or compromise on quality.
• **Full Quality**: No blurred group photos.
• **Easy Joining**: QR code or simple link access for seamless participation.
• **Built for Any Occasion**: Perfect for weddings, birthdays, trips, reunions, and team events.

## Data Safety and Security:

We take pride in maintaining the highest standards of data security:
No shared data with third-party partners
Data encryption during transmission ensuring total confidentiality
You can learn more about our privacy policy to better inform your decision on how we collect, share and utilize personal info.

In [107]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("OneAlbum", "https://onealbum.app")

Selecting relevant links for https://onealbum.app by calling llama3.2
Found 3 relevant links


# OneAlbum Brochure


## Welcome to OneAlbum


OneAlbum is a revolutionary platform that brings everyone's photos from an event into one shared album. Imagine having every photo, every angle, and every moment collected in one place — it's the future of event photography.


## Our Mission


We're on a mission to make capturing life's memories easier, more convenient, and fun for everyone. By bringing people together with OneAlbum, we aim to create a shared experience that lasts long after the event ends.


## How It Works


OneAlbum is easy to use and takes just four steps:

1. Create an event album in seconds.
2. Invite guests via QR code and they join instantly.
3. Everyone uploads their photos and videos from their own camera roll.
4. Relive together, every perspective, in one place.


## What We Offer


• Easy-to-use platform
• Automatic photo collection from everyone's devices
• Shareable albums for events of all kinds (weddings, trips, parties)
• Various filters to enhance the album experience

## Our Values:


#### Creativity:
We encourage creativity and self-expression through our easy-to-use platform.

#### Connection:
We're dedicated to bringing people together and fostering connection among event attendees.

#### Ease of use:
Our goal is to make capturing life's memories convenient and accessible for everyone.


#### Customer First:
We put the needs of our customers first, ensuring their albums are always up-to-date and fun to explore.


## Our Community


OneAlbum has been used at numerous events and occasions. Some notable examples include:

Sarah's Birthday Bash (NYC)
Emma & Liam's Wedding (Napa Valley)
Lake House Weekend

### Join the Fun!


If you're interested in seeing OneAlbum in action, please visit our website and sign up for an account.


#### Career Opportunities:


Join us to be a part of innovative technology that brings people closer together.

Discover your role at [www.onealbum.com](http://www.onealbum.com)


## Get in Touch


For inquiries or collaboration opportunities, contact:

Email: [support@onealbum.com](mailto:support@onealbum.com)
Phone: +1 (123) 456-7890

#### Facebook,
#### Instagram

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>